# Music Transformer — Training Notebook

**Architecture:** Huang et al. (ICLR 2019) — Relative self-attention  
**Key difference from Step 6:** No GPT-2, no absolute positional encodings.  
This model learns to attend to *relative distances* between tokens, which naturally  
captures musical repetition, phrases, and motifs.

### Two-stage training
1. **Pre-train** on MAESTRO (1,276 files, western classical piano) → general music model  
2. **Fine-tune** × 4 cultural traditions × 2 tokenisers (REMI / EC-REMI) → 8 culture-specific models

### Western classical is the baseline
MAESTRO *is* western classical — the pre-trained model is already the western classical model.  
No fine-tuning is done for western classical; it is evaluated directly from the pre-train checkpoint.  
The MAESTRO test split (10% = ~128 files) is held out and **never seen during pre-training**.

### Architecture
- 4 decoder layers, d_model=256, 4 heads, d_ff=1024 (~5M params)
- Relative multi-head self-attention with efficient skewing algorithm
- Trained entirely from scratch — custom PyTorch, no HuggingFace Trainer

### Train / Val / Test split
- **MAESTRO**: deterministic 80/10/10 file-level split (saved to JSON). Pre-train uses 80%+10% (90%). Test 10% is held out.  
- **Cultural traditions**: 80/10/10 per tradition via `get_split_files()`.

**Runtime:** T4 GPU recommended. Pre-training ~45 min, each fine-tune ~10-15 min.

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q miditok==3.0.4 symusic==0.5.6
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Cell 2 — Mount Drive and Clone Repo

Replace `REPO_URL` with your GitHub repo URL before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, os

REPO_URL  = 'https://github.com/AshrafZohdi/Thesis-Best.git'
REPO_DIR  = '/content/Thesis-Best'
DRIVE_OUT = '/content/drive/MyDrive/thesis_music_transformer'
os.makedirs(DRIVE_OUT, exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

print('Repo ready:', REPO_DIR)
print('Drive output:', DRIVE_OUT)

## Cell 3 — Imports and Path Setup

In [ ]:
import sys, math, json, random, time, shutil
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

ROOT = Path(REPO_DIR)

# ── Link Drive data into the repo ────────────────────────────────────────────
# The cloned repo has a data/ dir (metadata CSVs) but no MIDI files.
# Replace it with a symlink to Drive so training sees the full dataset.
DRIVE_DATA = Path('/content/drive/MyDrive/Thesis-Data/data')
REPO_DATA  = ROOT / 'data'
if not REPO_DATA.is_symlink():
    if REPO_DATA.exists():
        shutil.rmtree(REPO_DATA)   # remove cloned data/ (metadata CSVs only)
    REPO_DATA.symlink_to(DRIVE_DATA)
    print(f'Linked: {DRIVE_DATA} → {REPO_DATA}')
else:
    print(f'Symlink ready: {REPO_DATA} → {REPO_DATA.resolve()}')

# Verify MIDI files are accessible
print('\nMIDI file counts per tradition:')
for trad in ['western_classical', 'hindustani', 'carnatic', 'irish_folk', 'turkish_makam']:
    midi_dir = REPO_DATA / 'processed' / trad / 'midi'
    n = len(list(midi_dir.glob('*.mid')) + list(midi_dir.glob('*.midi'))) if midi_dir.exists() else 0
    status = '✓' if n > 0 else '✗ NOT FOUND'
    print(f'  {trad:<25} {n} files  {status}')

# ── sys.path setup ────────────────────────────────────────────────────────────
_ds = str(ROOT / 'datasets')
if _ds in sys.path:
    sys.path.remove(_ds)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from music_transformer.model import MusicTransformer, MusicTransformerConfig
from music_transformer.train import (
    TRADITIONS, CHUNK_SIZE, REMI_VOCAB_SIZE, EC_REMI_VOCAB_SIZE,
    get_split_files, MIDIChunkDataset, get_lr,
    load_remi_tokenizer, load_ec_remi_tokenizer,
)
# train.py adds ROOT/src to sys.path which shadows HuggingFace 'tokenizers'.
# Remove it now so miditok can find the real package.
_src = str(ROOT / 'src')
if _src in sys.path:
    sys.path.remove(_src)

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SPLIT_DIR = ROOT / 'music_transformer' / 'splits'
CKPT_DIR  = Path(DRIVE_OUT) / 'checkpoints'
GEN_DIR   = Path(DRIVE_OUT) / 'generated'
for d in [CKPT_DIR, GEN_DIR, SPLIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'\nDevice      : {DEVICE}')
print(f'Checkpoints : {CKPT_DIR}')
print(f'Generated   : {GEN_DIR}')

## Cell 4 — Data Split Inspection

This creates the 80/10/10 file-level split on first run and saves it to `music_transformer/splits/`.  
The split is deterministic (fixed seed) and stable across runs.

In [ ]:
# western_classical uses a separate MAESTRO split created in Cell 6 (pretrain).
# Only cultural traditions use the standard processed/midi/ split here.
CULTURAL_TRADITIONS_SPLIT = [t for t in TRADITIONS if t != 'western_classical']

print('=== Train / Val / Test split (cultural traditions) ===')
print(f'{"Tradition":<25} {"Train":>6} {"Val":>5} {"Test":>5}')
print('-' * 45)
for trad in CULTURAL_TRADITIONS_SPLIT:
    tr = get_split_files(trad, 'train', SPLIT_DIR)
    va = get_split_files(trad, 'val',   SPLIT_DIR)
    te = get_split_files(trad, 'test',  SPLIT_DIR)
    print(f'{trad:<25} {len(tr):>6} {len(va):>5} {len(te):>5}')
print()
print('western_classical uses full MAESTRO (1,276 files) — split created in Cell 6.')

# Persist split definitions to Drive so they survive session resets
drive_split = Path(DRIVE_OUT) / 'splits'
drive_split.mkdir(exist_ok=True)
for f in SPLIT_DIR.glob('*.json'):
    shutil.copy(f, drive_split / f.name)
print(f'\nSplit files saved to {drive_split}')

## Cell 5 — Training Helper

In [ ]:
PRETRAIN_EPOCHS  = 30
FINETUNE_EPOCHS  = 20

def build_loaders(traditions, tok, tok_type, batch_size=16, tradition_label='western_classical'):
    train_files, val_files = [], []
    for trad in traditions:
        train_files.extend(get_split_files(trad, 'train', SPLIT_DIR))
        val_files.extend(  get_split_files(trad, 'val',   SPLIT_DIR))
    random.shuffle(train_files)
    random.shuffle(val_files)
    print(f'  Files : {len(train_files)} train / {len(val_files)} val')

    train_ds = MIDIChunkDataset(train_files, tok, tok_type, tradition_label,
                                chunk_size=CHUNK_SIZE, stride=CHUNK_SIZE // 2)
    val_ds   = MIDIChunkDataset(val_files,   tok, tok_type, tradition_label,
                                chunk_size=CHUNK_SIZE)
    print(f'  Chunks: {len(train_ds)} train / {len(val_ds)} val')

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size * 2,
                              shuffle=False, num_workers=2)
    return train_loader, val_loader


def run_training(model, train_loader, val_loader,
                 epochs=30, max_lr=3e-4, min_lr=3e-5,
                 warmup_steps=500, grad_clip=1.0, weight_decay=0.01,
                 save_dir=None, log_every=100):

    save_dir = Path(save_dir) if save_dir else None

    # ── Auto-resume from latest checkpoint ───────────────────────────────────
    history  = []
    best_val = float('inf')
    start_epoch = 1

    if save_dir:
        hist_path = save_dir / 'history.json'
        if hist_path.exists():
            history = json.loads(hist_path.read_text())
        if history:
            n_done = history[-1]['epoch']
            if n_done >= epochs:
                print(f'  Already complete ({n_done}/{epochs} epochs) — skipping.')
                return min(e['val_loss'] for e in history), history
            start_epoch = n_done + 1
            best_val    = min(e['val_loss'] for e in history)
            print(f'  Resuming from epoch {start_epoch}/{epochs}  (best_val={best_val:.4f})')

    optimizer   = torch.optim.AdamW(model.parameters(), lr=max_lr,
                                    betas=(0.9, 0.95), weight_decay=weight_decay)
    total_steps = len(train_loader) * epochs
    warmup      = min(warmup_steps, total_steps // 10)
    use_amp     = (DEVICE.type == 'cuda')
    scaler      = torch.cuda.amp.GradScaler(enabled=use_amp)

    # Offset step counter so LR schedule is correct after resume
    step = (start_epoch - 1) * len(train_loader)

    for epoch in range(start_epoch, epochs + 1):
        model.train()
        epoch_loss, t0 = 0.0, time.time()

        for i, (x, y) in enumerate(train_loader, 1):
            x, y = x.to(DEVICE), y.to(DEVICE)
            lr = get_lr(step, warmup, total_steps, max_lr, min_lr)
            for pg in optimizer.param_groups:
                pg['lr'] = lr

            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                _, loss = model(x, targets=y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()
            step += 1

            if i % log_every == 0:
                print(f'  epoch {epoch:02d} | step {i:04d}/{len(train_loader)} '
                      f'| loss={epoch_loss/i:.4f} | lr={lr:.2e}')

        # Validation
        model.eval()
        vl = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                with torch.cuda.amp.autocast(enabled=use_amp):
                    _, loss = model(x.to(DEVICE), targets=y.to(DEVICE))
                vl += loss.item()
        vl /= len(val_loader)

        tl  = epoch_loss / len(train_loader)
        ppl = math.exp(min(vl, 10))
        print(f'Epoch {epoch:02d}/{epochs} | train={tl:.4f} val={vl:.4f} ppl={ppl:.1f} '
              f'({time.time()-t0:.0f}s)')
        history.append({'epoch': epoch, 'train_loss': tl, 'val_loss': vl, 'ppl': ppl})

        if save_dir:
            model.save(save_dir / 'latest')
            if vl < best_val:
                best_val = vl
                model.save(save_dir / 'best')
                print(f'  ✓ New best val={best_val:.4f}')
            # Write history after every epoch so a mid-run crash still has progress
            with open(save_dir / 'history.json', 'w') as f:
                json.dump(history, f, indent=2)

    return best_val, history

print('Training helpers ready.')


## Cell 6 — Stage 1: Pre-train on MAESTRO (80/10/10 split — test held out)

Train the Music Transformer on the **complete MAESTRO piano corpus**.  
The western classical test set (10%) is held out **before** pre-training begins — it is never seen.  
Pre-training uses train (80%) + val (10%) = 90% of files.

The test split is saved to `splits/western_classical_maestro.json` so evaluation can  
verify which files were truly held out.

Skip this cell if `checkpoints/pretrain/best/` already exists in Drive.

In [ ]:
PRETRAIN_DIR       = CKPT_DIR / 'pretrain'
MAESTRO_DIR        = Path('/content/drive/MyDrive/Thesis-Data/datasets/western_classical/maestro-v3.0.0')
MAESTRO_SPLIT_FILE = SPLIT_DIR / 'western_classical_maestro.json'

# Check how many pretrain epochs are already done
pretrain_hist_path = PRETRAIN_DIR / 'history.json'
pretrain_history   = json.loads(pretrain_hist_path.read_text()) if pretrain_hist_path.exists() else []
n_pretrain_done    = pretrain_history[-1]['epoch'] if pretrain_history else 0

if n_pretrain_done >= PRETRAIN_EPOCHS:
    print(f'Pre-training complete ({n_pretrain_done}/{PRETRAIN_EPOCHS} epochs) — skipping.')
    if MAESTRO_SPLIT_FILE.exists():
        maestro_split = json.loads(MAESTRO_SPLIT_FILE.read_text())
        print(f'MAESTRO split: {len(maestro_split["train"])} train / '
              f'{len(maestro_split["val"])} val / {len(maestro_split["test"])} test (held out)')
else:
    if n_pretrain_done > 0:
        print(f'=== Stage 1: Resuming pre-training ({n_pretrain_done}/{PRETRAIN_EPOCHS} epochs done) ===')
    else:
        print('=== Stage 1: Pre-training on full MAESTRO dataset ===')

    # ── Build or load MAESTRO split ───────────────────────────────────────────
    if MAESTRO_SPLIT_FILE.exists():
        maestro_split = json.loads(MAESTRO_SPLIT_FILE.read_text())
        train_files = [Path(f) for f in maestro_split['train']]
        val_files   = [Path(f) for f in maestro_split['val']]
        test_files  = [Path(f) for f in maestro_split['test']]
        print(f'Loaded split: {len(train_files)} train / {len(val_files)} val / {len(test_files)} test')
    else:
        maestro_files = sorted(
            list(MAESTRO_DIR.rglob('*.midi')) + list(MAESTRO_DIR.rglob('*.mid'))
        )
        print(f'Found {len(maestro_files)} MAESTRO files')
        if not maestro_files:
            raise FileNotFoundError(
                f'No MIDI files found at {MAESTRO_DIR}\nCheck that your Drive path is correct.'
            )
        random.seed(42)
        shuffled = maestro_files.copy()
        random.shuffle(shuffled)
        n          = len(shuffled)
        n_test     = max(1, int(0.10 * n))
        n_val      = max(1, int(0.10 * n))
        test_files  = shuffled[:n_test]
        val_files   = shuffled[n_test:n_test + n_val]
        train_files = shuffled[n_test + n_val:]
        print(f'Split: {len(train_files)} train / {len(val_files)} val / {len(test_files)} test (held out)')
        maestro_split = {
            'train': [str(f) for f in train_files],
            'val'  : [str(f) for f in val_files],
            'test' : [str(f) for f in test_files],
        }
        MAESTRO_SPLIT_FILE.write_text(json.dumps(maestro_split, indent=2))
        shutil.copy(MAESTRO_SPLIT_FILE, Path(DRIVE_OUT) / 'splits' / MAESTRO_SPLIT_FILE.name)
        print(f'Split saved → {MAESTRO_SPLIT_FILE.name}')

    # ── Build data loaders ────────────────────────────────────────────────────
    print('Tokenising MAESTRO files — please wait...')
    remi_tok = load_remi_tokenizer()
    train_ds = MIDIChunkDataset(train_files, remi_tok, 'remi', 'western_classical',
                                chunk_size=CHUNK_SIZE, stride=CHUNK_SIZE // 2)
    val_ds   = MIDIChunkDataset(val_files,   remi_tok, 'remi', 'western_classical',
                                chunk_size=CHUNK_SIZE)
    print(f'Chunks: {len(train_ds):,} train / {len(val_ds):,} val')
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)

    # ── Create or resume model ────────────────────────────────────────────────
    latest_ckpt = PRETRAIN_DIR / 'latest'
    if n_pretrain_done > 0 and (latest_ckpt / 'model.pt').exists():
        print(f'Loading latest checkpoint (epoch {n_pretrain_done})...')
        model = MusicTransformer.load(latest_ckpt, map_location='cpu').to(DEVICE)
    else:
        cfg = MusicTransformerConfig(
            vocab_size=REMI_VOCAB_SIZE, d_model=256, n_heads=4,
            n_layers=4, d_ff=1024, max_seq_len=CHUNK_SIZE, dropout=0.1,
        )
        model = MusicTransformer(cfg).to(DEVICE)
    print(f'Model: {model.n_params:,} parameters')

    best_val, history = run_training(
        model, train_loader, val_loader,
        epochs=PRETRAIN_EPOCHS, max_lr=3e-4, min_lr=3e-5,
        warmup_steps=1000,
        save_dir=PRETRAIN_DIR,
    )
    print(f'\nPre-training done. Best val_loss={best_val:.4f}  PPL={math.exp(min(best_val, 10)):.1f}')
    del model
    torch.cuda.empty_cache()


## Cell 7 — Stage 2: Fine-tune (4 Cultural Traditions × 2 Tokenisers = 8 Models)

Western classical is **skipped** — the pre-trained MAESTRO model already IS the western classical model.  
It is evaluated directly from the pre-train checkpoint using the held-out MAESTRO test split.

Each cultural model starts from the pre-trained checkpoint.  
For EC-REMI, the vocab is expanded from 284 → 526 tokens before fine-tuning  
(the 242 new cultural token embeddings are randomly initialised).

Models already checkpointed in Drive are skipped automatically.

In [ ]:
from miditok.classes import TokSequence
from symusic import Score as SScore

PRETRAIN_BEST = CKPT_DIR / 'pretrain' / 'best'
assert (PRETRAIN_BEST / 'model.pt').exists(), \
    f'Run Cell 6 first — pre-trained checkpoint not found at {PRETRAIN_BEST}'

# Western classical is not fine-tuned — it uses the pretrain checkpoint directly
CULTURAL_TRADITIONS = [t for t in TRADITIONS if t != 'western_classical']

results = {}

for tradition in CULTURAL_TRADITIONS:
    for tok_type in ['remi', 'ec_remi']:
        run_name = f'{tradition}_{tok_type}'
        save_dir = CKPT_DIR / run_name

        ft_hist_path = save_dir / 'history.json'
        ft_history   = json.loads(ft_hist_path.read_text()) if ft_hist_path.exists() else []
        n_ft_done    = ft_history[-1]['epoch'] if ft_history else 0
        if n_ft_done >= FINETUNE_EPOCHS:
            print(f'Skipping {run_name} — already complete ({n_ft_done}/{FINETUNE_EPOCHS} epochs).')
            best = min((e['val_loss'] for e in ft_history), default=float('nan'))
            results[run_name] = {'best_val_loss': best, 'ppl': math.exp(min(best, 10))}
            continue

        print(f'\n=== Fine-tuning: {run_name} ===')

        if tok_type == 'remi':
            tok = load_remi_tokenizer()
            vocab_size = REMI_VOCAB_SIZE
        else:
            tok, _ = load_ec_remi_tokenizer(tradition)
            vocab_size = EC_REMI_VOCAB_SIZE

        train_loader, val_loader = build_loaders(
            [tradition], tok, tok_type,
            batch_size=16, tradition_label=tradition
        )

        # Load model: resume from latest fine-tune ckpt if available, else from pretrain
        latest_ft = save_dir / 'latest'
        if n_ft_done > 0 and (latest_ft / 'model.pt').exists():
            print(f'  Resuming fine-tune from epoch {n_ft_done} checkpoint')
            model = MusicTransformer.load(latest_ft, map_location='cpu')
        else:
            model = MusicTransformer.load(PRETRAIN_BEST, map_location='cpu')
            if tok_type == 'ec_remi':
                model.resize_vocab(EC_REMI_VOCAB_SIZE)
                print(f'  Expanded vocab: {REMI_VOCAB_SIZE} → {EC_REMI_VOCAB_SIZE}')
        model = model.to(DEVICE)

        best_val, history = run_training(
            model, train_loader, val_loader,
            epochs=20, max_lr=1e-4, min_lr=1e-5,
            warmup_steps=100, save_dir=save_dir,
        )
        results[run_name] = {'best_val_loss': best_val, 'ppl': math.exp(min(best_val, 10))}

        del model
        torch.cuda.empty_cache()

# Print summary
print('\n=== Fine-tuning Summary (4 cultural traditions × 2 tokenisers) ===')
print(f'{"Model":<35} {"Val Loss":>10} {"PPL":>8}')
print('-' * 57)
for run, r in sorted(results.items()):
    print(f'{run:<35} {r["best_val_loss"]:>10.4f} {r["ppl"]:>8.1f}')

with open(CKPT_DIR / 'finetune_summary.json', 'w') as f:
    json.dump(results, f, indent=2)

## Cell 8 — Generate MIDI Samples

For each model, generate 5 MIDI files seeded from the **test split**.  
- **Western classical** (REMI only): uses the pre-train checkpoint + held-out MAESTRO test files  
- **Cultural traditions** (REMI + EC-REMI): uses the fine-tune checkpoint + tradition test files

In [ ]:
import pretty_midi
from miditok.classes import TokSequence
from symusic import Score as SScore

N_SAMPLES   = 20
GEN_TOKENS  = 512
SEED_LEN    = 64
TEMPERATURE = 0.92
TOP_P       = 0.92

def generate_with_rep_penalty(model, prompt, max_new_tokens,
                               temperature=0.92, top_p=0.92,
                               rep_penalty=1.3, rep_window=64):
    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            ctx = prompt if prompt.size(1) <= model.cfg.max_seq_len else prompt[:, -model.cfg.max_seq_len:]
            logits, _ = model(ctx)
            logits = logits[:, -1, :] / temperature
            if rep_penalty > 1.0:
                recent = prompt[0, -rep_window:].unique()
                logits[0, recent] /= rep_penalty
            probs = torch.nn.functional.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, dim=-1, descending=True)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_probs[(cumulative - sorted_probs) > top_p] = 0.0
            sorted_probs.div_(sorted_probs.sum(dim=-1, keepdim=True) + 1e-9)
            next_token = sorted_idx.gather(-1, torch.multinomial(sorted_probs, 1))
            prompt = torch.cat([prompt, next_token], dim=1)
    return prompt

remi_dec = load_remi_tokenizer()


def get_seed_ids(tradition, tok, tok_type, idx=0, override_files=None):
    try:
        files = override_files if override_files is not None else get_split_files(tradition, 'test', SPLIT_DIR)
        if not files:
            return None
        fpath = Path(files[idx % len(files)])
        if tok_type == 'ec_remi':
            tokens = tok.tokenize(fpath, tradition=tradition)
            if tokens and len(tokens) >= SEED_LEN:
                return tok.encode(tokens)[:SEED_LEN]
        else:
            seqs = tok.encode(SScore(str(fpath)))
            if seqs and len(seqs[0].ids) >= SEED_LEN:
                return seqs[0].ids[:SEED_LEN]
    except Exception as e:
        print(f'    seed error: {e}')
    return None


# Cents per unit for each microtonal system
_CENTS_PER_COMMA_T  = 1200.0 / 53          # Turkish 53-TET ≈ 22.6 cents
_CENTS_PER_SHRUTI_I = 1200.0 / (22 * 2)    # Indian shruti half ≈ 27.3 cents
_PB_SCALE           = 8191 / 200.0         # pitch-bend units per cent (±200 cent range)

def ec_remi_ids_to_midi(ids, out_path, ec_tok):
    """
    Decode EC-REMI IDs to MIDI, faithfully applying microtonal pitch bends
    from MicroOffset tokens so EC-REMI sounds audibly different from plain REMI.
    """
    import tempfile, os
    try:
        tokens = ec_tok.decode(ids)

        # ── Base MIDI from REMI subsequence ──────────────────────────────────
        remi_ids = [i for i in ids if 0 <= i < ec_tok.remi_vocab_size]
        if len(remi_ids) < 5:
            return False
        score = ec_tok._remi.decode([TokSequence(ids=remi_ids)])
        if not score.tracks or not any(len(t.notes) > 0 for t in score.tracks):
            return False

        # Write to temp file so pretty_midi can reload it
        tmp = str(out_path) + '._tmp.mid'
        score.dump_midi(tmp)

        pm = pretty_midi.PrettyMIDI(tmp)
        os.unlink(tmp)

        # ── Parse MicroOffset tokens and map to note indices ──────────────────
        note_offsets: list[tuple[str, int]] = []  # (prefix, offset_val) per Pitch token
        i = 0
        while i < len(tokens):
            tok = tokens[i]
            if tok.startswith('Pitch_'):
                prefix, val = '', 0
                j = i + 1
                while j < len(tokens) and tokens[j].startswith(('MicroOffset_', 'Ornament_', 'Modal_')):
                    t = tokens[j]
                    if t.startswith('MicroOffset_T_') or t.startswith('MicroOffset_I_'):
                        try:
                            prefix = 'T' if '_T_' in t else 'I'
                            val    = int(t.split('_')[-1])
                        except ValueError:
                            pass
                    j += 1
                note_offsets.append((prefix, val))
            i += 1

        # ── Apply pitch bends to pretty_midi instrument ────────────────────────
        if note_offsets and pm.instruments:
            inst = pm.instruments[0]
            notes = sorted(inst.notes, key=lambda n: n.start)
            new_bends = []
            for n_idx, note in enumerate(notes):
                if n_idx >= len(note_offsets):
                    break
                pfx, val = note_offsets[n_idx]
                if val == 0 or not pfx:
                    continue
                cents    = val * (_CENTS_PER_COMMA_T if pfx == 'T' else _CENTS_PER_SHRUTI_I)
                pb_val   = int(max(-8191, min(8191, cents * _PB_SCALE)))
                onset    = max(0.0, note.start - 0.005)
                reset_t  = note.start + note.get_duration() * 0.6
                new_bends.append(pretty_midi.PitchBend(pitch=pb_val,  time=onset))
                new_bends.append(pretty_midi.PitchBend(pitch=0,        time=reset_t))
            inst.pitch_bends = sorted(inst.pitch_bends + new_bends, key=lambda pb: pb.time)

        pm.write(str(out_path))
        return True
    except Exception as e:
        print(f'    decode error: {e}')
        return False


def ids_to_midi(ids, out_path):
    try:
        valid = [i for i in ids if 0 <= i < REMI_VOCAB_SIZE]
        if len(valid) < 5:
            return False
        score = remi_dec.decode([TokSequence(ids=valid)])
        if not score.tracks or not any(len(t.notes) > 0 for t in score.tracks):
            return False
        score.dump_midi(str(out_path))
        return True
    except Exception as e:
        print(f'    decode error: {e}')
        return False


gen_summary = {}

# ── Western classical: generate from pretrain checkpoint (REMI only) ──────────
# Restore MAESTRO split from Drive if not present in local SPLIT_DIR
if not MAESTRO_SPLIT_FILE.exists():
    drive_split_backup = Path(DRIVE_OUT) / 'splits' / 'western_classical_maestro.json'
    if drive_split_backup.exists():
        SPLIT_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copy(drive_split_backup, MAESTRO_SPLIT_FILE)
        print('Restored MAESTRO split from Drive')

maestro_test = None
if MAESTRO_SPLIT_FILE.exists():
    maestro_test = json.loads(MAESTRO_SPLIT_FILE.read_text())['test']

print('=== Western Classical (from pretrain checkpoint) ===')
wc_ckpt = CKPT_DIR / 'pretrain' / 'best'
if (wc_ckpt / 'model.pt').exists() and maestro_test:
    remi_tok = load_remi_tokenizer()
    model    = MusicTransformer.load(wc_ckpt, map_location='cpu').to(DEVICE)
    out_dir  = GEN_DIR / 'western_classical_remi'
    out_dir.mkdir(parents=True, exist_ok=True)
    saved = 0
    for i in range(N_SAMPLES):
        seed = get_seed_ids('western_classical', remi_tok, 'remi',
                            idx=i, override_files=maestro_test)
        if seed is None:
            print(f'  [{i}] no seed')
            continue
        torch.manual_seed(42 + i)
        prompt = torch.tensor([seed], dtype=torch.long, device=DEVICE)
        output = generate_with_rep_penalty(model, prompt, GEN_TOKENS, TEMPERATURE, TOP_P)
        ids    = output[0].cpu().tolist()
        ok     = ids_to_midi(ids, out_dir / f'sample_{i:02d}.mid')
        print(f'  [{i}] {len(ids)} tokens → {"saved" if ok else "decode failed"}')
        if ok:
            saved += 1
    gen_summary['western_classical_remi'] = {'saved': saved, 'total': N_SAMPLES}
    del model
    torch.cuda.empty_cache()
else:
    print('  Pretrain checkpoint or MAESTRO split not found — skipping.')

# ── Cultural traditions: generate from fine-tune checkpoints ─────────────────
for tradition in CULTURAL_TRADITIONS:
    for tok_type in ['remi', 'ec_remi']:
        run_name  = f'{tradition}_{tok_type}'
        ckpt_path = CKPT_DIR / run_name / 'best'

        if not (ckpt_path / 'model.pt').exists():
            print(f'Skipping {run_name} — no checkpoint')
            continue

        print(f'\nGenerating: {run_name}')
        model = MusicTransformer.load(ckpt_path, map_location='cpu').to(DEVICE)
        tok   = load_remi_tokenizer() if tok_type == 'remi' else load_ec_remi_tokenizer(tradition)[0]
        out_dir = GEN_DIR / run_name
        out_dir.mkdir(parents=True, exist_ok=True)

        saved = 0
        for i in range(N_SAMPLES):
            seed = get_seed_ids(tradition, tok, tok_type, idx=i)
            if seed is None:
                print(f'  [{i}] no seed')
                continue
            torch.manual_seed(42 + i)
            prompt = torch.tensor([seed], dtype=torch.long, device=DEVICE)
            # Use slightly lower top_p for data-scarce traditions to reduce looping
            _top_p = 0.85 if tradition in ('hindustani', 'carnatic') else TOP_P
            output = generate_with_rep_penalty(model, prompt, GEN_TOKENS, TEMPERATURE, _top_p)
            ids    = output[0].cpu().tolist()
            if tok_type == 'ec_remi':
                ok = ec_remi_ids_to_midi(ids, out_dir / f'sample_{i:02d}.mid', tok)
            else:
                ok = ids_to_midi(ids, out_dir / f'sample_{i:02d}.mid')
            print(f'  [{i}] {len(ids)} tokens → {"saved" if ok else "decode failed"}')
            if ok:
                saved += 1

        gen_summary[run_name] = {'saved': saved, 'total': N_SAMPLES}
        del model
        torch.cuda.empty_cache()

print('\n=== Generation Summary ===')
for run, s in gen_summary.items():
    print(f'  {run:<35} {s["saved"]}/{s["total"]} MIDI files')

## Cell 9 — Save Split Definitions and Print Final Report

In [ ]:
# Persist split definitions to Drive
drive_split = Path(DRIVE_OUT) / 'splits'
drive_split.mkdir(exist_ok=True)
for f in SPLIT_DIR.glob('*.json'):
    shutil.copy(f, drive_split / f.name)

# ── Final report ──────────────────────────────────────────────────────────────
pretrain_history = []
pretrain_best_val = float('nan')
pretrain_history_path = CKPT_DIR / 'pretrain' / 'history.json'
if pretrain_history_path.exists():
    pretrain_history = json.loads(pretrain_history_path.read_text())
    pretrain_best_val = min(e['val_loss'] for e in pretrain_history)

finetune_results = {}
summary_path = CKPT_DIR / 'finetune_summary.json'
if summary_path.exists():
    finetune_results = json.loads(summary_path.read_text())

print('=== Music Transformer — Final Results ===')
print()
print(f'{"Model":<35} {"Stage":<12} {"Val Loss":>10} {"PPL":>8} {"Ckpt":>6}')
print('-' * 75)

# Western classical — pretrain checkpoint
wc_ok  = (CKPT_DIR / 'pretrain' / 'best' / 'model.pt').exists()
wc_ppl = math.exp(min(pretrain_best_val, 10)) if not math.isnan(pretrain_best_val) else float('nan')
print(f'{"western_classical_remi":<35} {"pretrain":<12} {pretrain_best_val:>10.4f} {wc_ppl:>8.1f} {"✓" if wc_ok else "✗":>6}')
print(f'{"western_classical_ec_remi":<35} {"(not done)":<12} {"—":>10} {"—":>8} {"—":>6}')
print()

# Cultural fine-tune results
for tradition in CULTURAL_TRADITIONS:
    for tok_type in ['remi', 'ec_remi']:
        run = f'{tradition}_{tok_type}'
        r   = finetune_results.get(run, {})
        ok  = (CKPT_DIR / run / 'best' / 'model.pt').exists()
        vl  = r.get('best_val_loss', float('nan'))
        ppl = r.get('ppl', float('nan'))
        print(f'{run:<35} {"finetune":<12} {vl:>10.4f} {ppl:>8.1f} {"✓" if ok else "✗":>6}')

print()
print('=== Test Split (held out — never trained on) ===')
if MAESTRO_SPLIT_FILE.exists():
    wc_test = json.loads(MAESTRO_SPLIT_FILE.read_text())['test']
    print(f'  {"western_classical":<25} {len(wc_test)} MAESTRO files')
for trad in CULTURAL_TRADITIONS:
    files = get_split_files(trad, 'test', SPLIT_DIR)
    print(f'  {trad:<25} {len(files)} files')

print(f'\nAll outputs saved to: {DRIVE_OUT}')